# The Registry Browser

pygeodata keeps a **registry** — a structured index of every cached output it
has produced, including the parameters, state hashes, spec, and file location
for each entry. The **registry browser** is a local web UI that lets you
navigate this index interactively.

This notebook covers:

- Launching the browser from Python
- The entry catalog: what gets listed and how to filter it
- Inspecting an individual entry (params, spec, co-outputs, graph)
- When the registry is useful

In [ ]:
import os, sys
os.chdir('../../..')
sys.path.insert(0, 'docs/loaders')

In [ ]:
from pathlib import Path

from pygeodata import SpatialSpec, get_config, process

get_config().update(
    path_cache=Path('data/processed'),
    path_figures=Path('data/figures'),
)

spec = SpatialSpec.from_raster_file('data/wtd.tif')

## 1. Populate the cache

Run a few loaders from the tutorial pipeline so the registry has something
to show. Skip this if you have already run previous notebooks.

In [ ]:
from pipeline import (
    ElevationLoader,
    WaterTableDepthLoader,
    CountryMaskLoader,
    LandWaterTableDepth,
    MeanStdLoader,
    WTDFigure,
    WTDStatsLoader,
)

wtd  = WaterTableDepthLoader()
mask = CountryMaskLoader()

# A few representative entries
loaders = [
    ElevationLoader(),
    wtd,
    mask,
    LandWaterTableDepth(wtd=wtd, mask=mask),
    MeanStdLoader(wtd=wtd, mask=mask, stat='mean'),
    WTDFigure(),
    WTDStatsLoader(wtd=wtd, mask=mask),
]

for ldr in loaders:
    process(ldr, spec)

print('Done — cache populated.')

## 2. Launch the registry browser

`open_registry_browser()` starts a local Flask server on a random free port
and opens the URL in your default browser. The call blocks — use it from a
terminal rather than a notebook if you want to keep running cells.

```python
from pygeodata.registry_browser import open_registry_browser
open_registry_browser()
```

The function prints the URL before opening the browser, so you can also paste
it into any browser manually:

```
http://127.0.0.1:52341
```

The server reads from `path_cache` and `path_figures` as configured in the
current session. No data leaves your machine.

![Registry browser — entry catalog](../_static/registry/catalog.png)

*The catalog lists all cached entries. The sidebar filters by class; the green dot next to each class name indicates all entries are valid (hashes match).*

## 3. The entry catalog

The landing page shows all cached entries, grouped by loader class. Each row
shows:

| Column | Description |
|--------|-------------|
| **Class** | Loader class name (links to the class card) |
| **Spec** | CRS, resolution, shape and bounding box of the output |
| **Parameters** | Key–value pairs that identify this specific output |
| **File** | The cache file path (click to open in Finder/Explorer) |
| **Valid** | Whether the current code and params match the stored hash |

### Filtering

Use the search box to filter by class name or parameter value. The class
sidebar on the left lets you show only entries from a specific class.

### Cache validity indicator

The **Valid** column turns red when the stored state hash no longer matches
the live hash — meaning the class code or dependencies have changed since the
output was written. These entries will be reprocessed on the next `load()` or
`process()` call.

## 4. Inspecting an entry

Click any row in the catalog to open the **entry detail panel**.

### Params card
Shows the full parameter dict as stored in the `.params.json` file — the same
dict returned by `loader.get_params_as_json(spec=spec)`. Nested loaders appear
as expandable sub-trees.

### Spec card
Shows CRS, affine transform, shape, resolution, and bounding box for this
specific cached output.

### Co-outputs card
When an entry was produced by a co-output `_process` (notebook 05), the
Co-outputs card lists all siblings that were written in the same run. Each
sibling links to its own entry card — useful for comparing the mean and std
rasters side by side.

### Dependency graph
The graph icon in the top-right of an entry card opens an interactive
dependency graph for that loader class. Nodes are clickable — clicking a node
jumps to the entry for that upstream loader (if it is cached).

### Source code popup
The source icon shows the full source of the loader class as pygeodata sees it
— the same source used for the dependency tree hash. If you suspect a
cache-invalidation mystery, comparing the live source popup against what you
see in your editor is the first thing to check.

![Entry detail — LandWaterTableDepth](../_static/registry/entry_detail.png)

*Detail panel for `LandWaterTableDepth`: CRS, resolution, shape, bounds, file path (with Reveal/Copy buttons), associated upstream entries (CountryMaskLoader and WaterTableDepthLoader), and the full parameter tree.*

## 5. The class catalog

The **Classes** tab lists every loader class that has at least one cached
entry. Each class card shows:

- The class docstring
- Which parameters are present across all entries
- A link to the class-level dependency graph
- A count of valid / invalid / total cached entries

This view is useful when you want to understand the structure of your pipeline
rather than look at individual outputs.

![Co-outputs — MeanStdLoader](../_static/registry/co_outputs.png)

*`MeanStdLoader` was produced by a co-output `_process`. The CO-OUTPUTS card lists both siblings (mean and std), each with its own file path, Reveal button, and distinguishing parameter (`STAT mean` vs `STAT std`). Both share the same state hash since they were written in one run.*

## 6. When the registry browser is useful

**Debugging invalid caches** — the valid/invalid indicator pinpoints which
entries are stale after a code change, and the source popup shows exactly what
changed.

**Auditing outputs** — the params and spec cards give a full provenance record
for any cached file, without needing to re-run the code that produced it.

**Exploring a pipeline** — the graph view and class catalog give a quick
overview of a project's structure, even for code you didn't write.

**Finding co-outputs** — when one `_process` produces multiple files, the
co-output card links them all, avoiding manual path hunting.

**Opening files directly** — the file column links open the output file in
your file manager (Reveal) or in a default application (Open), saving several
steps when you want to inspect a raster in QGIS.